<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/07_rate_again.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/rating_200")

Mounted at /content/drive


In [2]:
rating_sheet = pd.read_csv(folder / "rating_sheet.csv", dtype=str, keep_default_na=False)
my_ratings_path = folder / "my_ratings_v2.csv"

if my_ratings_path.exists():
    my_ratings = pd.read_csv(my_ratings_path, dtype=str, keep_default_na=False)
else:
    my_ratings = pd.DataFrame(columns=["file_id", "rating"])

print(len(my_ratings), "rows already rated")

0 rows already rated


In [3]:
rows_to_rate = 30

for i in range(rows_to_rate):
    file_id = rating_sheet.loc[i, "file_id"]
    if file_id in list(my_ratings["file_id"]):
        continue

    print("row", i + 1, "of", rows_to_rate)
    print("correct text:", rating_sheet.loc[i, "correct_text"])
    print("model text:  ", rating_sheet.loc[i, "model_text"])
    answer = input("1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: ")

    if answer == "q":
        break
    if answer not in ["1", "0", "?"]:
        print("wrong key, this row is skipped. Run this cell again to rate it.")
        continue

    my_ratings.loc[len(my_ratings)] = [file_id, answer]
    my_ratings.to_csv(my_ratings_path, index=False)
    print()

row 1 of 30
correct text: वुड्सको प्रदर्शन प्रभावशाली
model text:   हुटस्कव प्रदर्शन प्रभाव शाली
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 0

row 2 of 30
correct text: मन्दिरमा तीनवटा हिन्दू
model text:   मन्दिरमा तिनोटा हिन्दु
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 0

row 3 of 30
correct text: सङ्घर्षका कार्यक्रमहरू पनि
model text:   संगर्षका कार्यक्रमहरू पनि
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 0

row 4 of 30
correct text: उहाँ धौलागिरी अञ्चलबाट
model text:   उहाँ धलाग्री अञ्चलबाट
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 0

row 5 of 30
correct text: पहिलो लक्ष्य हुने
model text:   पहिलो लच्ष्य हुने
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 0

row 6 of 30
correct text: आफ्नो जिब्रो पड्काउन
model text:   आफ्नो जिब्रो पट्काउन
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 0

row 7 of 30
correct text: गाड्नुका साथै हाँगामा
model text:   गाडnुका साथे आँगामा
1 = acceptable, 0 = n

In [4]:
print(len(my_ratings), "rows rated")
print(my_ratings["rating"].value_counts())

30 rows rated
rating
0    27
1     3
Name: count, dtype: int64


In [5]:
old_ratings = pd.read_csv(folder / "my_ratings.csv", dtype=str, keep_default_na=False)
old_ratings = old_ratings.rename(columns={"rating": "old_rating"})

new_ratings = my_ratings.rename(columns={"rating": "new_rating"})
both = old_ratings.merge(new_ratings, on="file_id")
both = both.merge(rating_sheet[["file_id", "correct_text", "model_text"]], on="file_id")

same_count = (both["old_rating"] == both["new_rating"]).sum()
print("same rating:", same_count, "of", len(both))

changed = both[both["old_rating"] != both["new_rating"]]
changed[["correct_text", "model_text", "old_rating", "new_rating"]]

same rating: 28 of 30


,correct_text,model_text,old_rating,new_rating
12,उनलाई अन्ना हजारेका,उनलाई अन्न हजारेका,1,0
19,क्लिपटाउनको फ्रिडम चार्टर सम्मेलनमा,क्लिपटाउनको फ्रिडम चार्टर सम्मिलनमा,1,0


In [6]:
scores = pd.read_csv(folder / "scores_200.csv", dtype={"file_id": str})

unsure_count = (my_ratings["rating"] == "?").sum()
rated = my_ratings[my_ratings["rating"].isin(["0", "1"])].copy()
rated = rated.merge(scores[["file_id", "wer", "cer"]], on="file_id")
rated["rating"] = rated["rating"].astype(int)

print("rated rows:", len(rated), "| unsure rows:", unsure_count)
print(rated["rating"].value_counts())

rated rows: 30 | unsure rows: 0
rating
0    27
1     3
Name: count, dtype: int64


In [7]:
rated.groupby("rating")[["wer", "cer"]].mean().round(2)

,wer,cer
rating,,
0,0.79,0.24
1,0.61,0.07


In [8]:
print("WER and rating:", round(rated["rating"].corr(rated["wer"], method="spearman"), 2))
print("CER and rating:", round(rated["rating"].corr(rated["cer"], method="spearman"), 2))

WER and rating: -0.05
CER and rating: -0.29
